In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

from scipy import signal
from scipy import interpolate
from scipy import ndimage
import numpy as np
import pycatch22 
from sktime.transformations.panel import catch22
import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
load_dotenv = dotenv.load_dotenv('../.env')
# load local library
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
import extractor
import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

from time import sleep

from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq
from scipy.signal import cwt, ricker
import pymannkendall as mk

In [31]:
######################
### Main functions ###
######################

def get_crossectional(tsdf: pd.DataFrame, id_col='ID',
                      val_col='eGFR_CKDEpi2012', time_col='Time_col', 
                      catch22=False):

    CustomExtractor = extractor.Extractor()
    CustomExtractor.fit(tsdf, 
                        id_col=id_col, 
                        val_col=val_col, 
                        time_col=time_col)

    ts_data_agg = CustomExtractor.transform()

    FreshExtractor = extractor.TsFreshExtractor()
    FreshExtractor.fit(tsdf, 
                        id_col=id_col, 
                        val_col=val_col,
                        time_col=time_col)
    ts_data_agg_fresh = FreshExtractor.transform()


    if catch22==True:
        Catch22Extractor  = extractor.Catch22Extractor()
        Catch22Extractor.fit(tsdf, 
                    id_col=id_col, 
                    val_col=val_col,
                    time_col=time_col)
        ts_data_agg_catch22 = Catch22Extractor.transform()


    ts_data_agg = ts_data_agg.set_index('id')

    FINAL_FEATURES = ts_data_agg.merge(ts_data_agg_fresh,
                                       left_index=True,
                                       right_index=True)
    if catch22==True:
        FINAL_FEATURES = FINAL_FEATURES.merge(ts_data_agg_catch22,
                                              left_index=True,
                                              right_index=True)
    return FINAL_FEATURES

In [3]:
AKI_PATH = os.environ['AKI_PATH']
os.chdir(AKI_PATH)

In [4]:
ts_data = pd.read_parquet("ts.parquet.snappy")

In [5]:
ts_data = ts_data.assign(ID=ts_data.ID.astype('int64'))

## Exploration

In [6]:
min_days_length = 1*365
maxts = ts_data.groupby('ID').Time_days.max()
ltids = maxts[maxts>min_days_length].index

print(f'There are {ltids.shape[0]} patients (or {round(100*ltids.shape[0]/maxts.shape[0], 2)}%) with minimally {min_days_length} of measurements')

There are 7401 patients (or 36.22%) with minimally 365 of measurements


## Preprocess

In [7]:
RangeList = [90, 180] + list(np.arange(1*365,10*365, 365))
WINDOW_SIZE = 3
SMOOTHING_TYPE = 'gaussian_kernel' # rolling_mean or gaussian_kernel

RangeDFDict = {}
RangeDFDict_smoothed = {}
for MinDays in RangeList:
    ts_temp = preprocessing.get_filtered_df(ts_data.copy(), id_col='ID', time_col='Time_days', min_days=MinDays)
    
    print(f'Interpolating for min days: {MinDays}')
    RangeDFDict[MinDays] = preprocessing.get_interpolated(ts_temp, id_col='ID', time_col='Time_days',
                                              max_days=MinDays, val_col='eGFR_CKDEpi2012', time_res=7)
    
    print(f"Smoothing for min days: {MinDays} with window {WINDOW_SIZE}")
    if SMOOTHING_TYPE == 'rolling_mean':
        RangeDFDict_smoothed[MinDays] = preprocessing.get_smoothed_rolling_mean(RangeDFDict[MinDays],
                                                        id_col='ID', time_col='Time_days',
                                                        val_col='eGFR_CKDEpi2012', window=WINDOW_SIZE)
    elif SMOOTHING_TYPE == 'gaussian_kernel':
        RangeDFDict_smoothed[MinDays] = preprocessing.get_smoothed_gaussian_kernel(RangeDFDict[MinDays],
                                                        id_col='ID', time_col='Time_days',
                                                        val_col='eGFR_CKDEpi2012', window=WINDOW_SIZE)
    

Interpolating for min days: 90


100%|██████████| 9829/9829 [00:18<00:00, 541.41it/s]


Smoothing for min days: 90 with window 3


100%|██████████| 9829/9829 [00:13<00:00, 744.64it/s]


Interpolating for min days: 180


...

100%|██████████| 2534/2534 [00:07<00:00, 327.52it/s]


Interpolating for min days: 3285


100%|██████████| 2210/2210 [00:03<00:00, 650.53it/s]


Smoothing for min days: 3285 with window 3


100%|██████████| 2210/2210 [00:05<00:00, 368.70it/s]


In [28]:
features = {"bla" : zip(list(range(10)), list(range(10))),
            "blo" : zip(list(range(10)), list(range(10))),
        }

pd.DataFrame.from_dict(features,  orient='index').reset_index()

,index,0,1,2,3,4,5,6,7,8,9
0,bla,"(0, 0)","(1, 1)","(2, 2)","(3, 3)","(4, 4)","(5, 5)","(6, 6)","(7, 7)","(8, 8)","(9, 9)"
1,blo,"(0, 0)","(1, 1)","(2, 2)","(3, 3)","(4, 4)","(5, 5)","(6, 6)","(7, 7)","(8, 8)","(9, 9)"


In [33]:
CrossSectDict = {}
# get_smoothNsmooth_diffStatistics
for MinDays in RangeList:
    print(f'Getting cross-sectional features for min days, for the smoothed set, with periods of: {MinDays} days')
    tsS = pd.DataFrame.from_dict(RangeDFDict_smoothed[MinDays], orient='columns')
    smoothed_cross = get_crossectional(tsS, 
                                        id_col='ID',
                                        val_col='eGFR_CKDEpi2012',
                                        time_col='Time_days',
                                        catch22=True)
    ############
    print(f'Getting cross-sectional features for min days, for the raw set, with periods of: {MinDays} days')
    tsR = pd.DataFrame.from_dict(RangeDFDict[MinDays], orient='columns')
    raw_cross = get_crossectional(tsR, 
                                        id_col='ID',
                                        val_col='eGFR_CKDEpi2012',
                                        time_col='Time_days',
                                        catch22=True)
    ############
    print(f'Merging cross-sectional features for min days, the raw set, with periods of: {MinDays} days')
    merged_cross = smoothed_cross.merge(raw_cross,
                                        left_index=True, 
                                        right_index=True, 
                                        suffixes=('_smoothed', '_raw'))
    smoothNsmoothStats = extractor.get_smoothNsmooth_diffStatistics(tsS,tsR, 
                                                                    id_col='ID',
                                                                    val_col='eGFR_CKDEpi2012',
                                                                    time_col='Time_days')
    
    smoothed_cross = smoothed_cross.merge(smoothNsmoothStats, left_index=True, right_index=True)
    raw_cross = raw_cross.merge(smoothNsmoothStats, left_index=True, right_index=True)
    merged_cross = merged_cross.merge(smoothNsmoothStats, left_index=True, right_index=True)

    CrossSectDict[MinDays] = {
                              'smoothed': smoothed_cross,
                              'raw': raw_cross,
                              'merged': merged_cross
                            }
    
    gc.collect()

Getting cross-sectional features for min days, for the smoothed set, with periods of: 90 days


 39%|███▉      | 3813/9829 [00:21<00:52, 115.54it/s]\\ds\data\LAB\laupodteam\AIOS\Bram\production\TimEx\sandbox\..\src\extractor.py:45: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skewness = skew(ts_data)
\\ds\data\LAB\laupodteam\AIOS\Bram\production\TimEx\sandbox\..\src\extractor.py:46: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kurtosis_value = kurtosis(ts_data)
100%|██████████| 9829/9829 [00:06<00:00, 1566.52it/s]


Getting cross-sectional features for min days, for the raw set, with periods of: 90 days


 39%|███▉      | 3815/9829 [00:15<00:23, 257.90it/s]\\ds\data\LAB\laupodteam\AIOS\Bram\production\TimEx\sandbox\..\src\extractor.py:45: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skewness = skew(ts_data)
\\ds\data\LAB\laupodteam\AIOS\Bram\production\TimEx\sandbox\..\src\extractor.py:46: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kurtosis_value = kurtosis(ts_data)
100%|██████████| 9829/9829 [00:06<00:00, 1612.79it/s]


Merging cross-sectional features for min days, the raw set, with periods of: 90 days
Getting cross-sectional features for min days, for the smoothed set, with periods of: 180 days


...

Merging cross-sectional features for min days, the raw set, with periods of: 2920 days
Getting cross-sectional features for min days, for the smoothed set, with periods of: 3285 days


100%|██████████| 2210/2210 [00:12<00:00, 180.08it/s]


Getting cross-sectional features for min days, for the raw set, with periods of: 3285 days


100%|██████████| 2210/2210 [00:09<00:00, 228.27it/s]


Merging cross-sectional features for min days, the raw set, with periods of: 3285 days


## Remove redundant features

In [36]:
from sklearn.feature_selection import VarianceThreshold

print("Removing zero variance features for all periods...")
for period in CrossSectDict.keys():
    for key in CrossSectDict[period].keys():
        FINAL_FEATURES = CrossSectDict[period][key]
        
        var_thresh = VarianceThreshold(threshold=0.)
        var_thresh.fit(FINAL_FEATURES)
        variances = var_thresh.variances_

        # remove features with zero variance
        zero_variance_features = FINAL_FEATURES.columns[variances == 0]
        non_zero_variance_features = FINAL_FEATURES.columns[variances > 0]
        FINAL_FEATURES = FINAL_FEATURES.drop(zero_variance_features, axis=1)
        
        CrossSectDict[period][key] = FINAL_FEATURES    

Removing zero variance features for all periods...


## Imputation

In [37]:
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer

In [38]:
imputer = SimpleImputer(strategy='mean')

## Scale

In [39]:
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AffinityPropagation, SpectralClustering, HDBSCAN
from umap import UMAP
from scaler import GaussRankScaler
from sklearn.pipeline import Pipeline

In [40]:
#scaler = GaussRankScaler()
#scaler = QuantileTransformer(output_distribution='normal')
scaler = StandardScaler()
#reducer = PCA(n_components=100)
reducer = UMAP(n_components=6, n_neighbors=15, min_dist=0., metric='manhattan')

## Cluster

In [41]:
clusterer = HDBSCAN(min_cluster_size=20, min_samples=15, cluster_selection_epsilon=0.5)

# Run pipeline

In [48]:
ModelDict = {}
for period in CrossSectDict.keys():
    ModelDict[period] = {}
    for preptype in CrossSectDict[period].keys():
        le_pipe_clusterer = Pipeline([('imputer', imputer), 
                                      ('scaler', scaler),
                                      ('reducer', reducer), 
                                      ('clusterer', clusterer)], verbose=True)

        print(f"Running pipeline for period: {period}, prep type: {preptype}")
        le_pipe_clusterer.fit(CrossSectDict[period][preptype])
        CrossSectDict[period][preptype]['cluster'] = le_pipe_clusterer.named_steps['clusterer'].labels_
        CrossSectDict[period][preptype]['cluster_proba'] = le_pipe_clusterer.named_steps['clusterer'].probabilities_
        ModelDict[period][preptype] = le_pipe_clusterer

Running pipeline for period: 90, prep type: smoothed
[Pipeline] ........... (step 1 of 4) Processing imputer, total=   0.1s
[Pipeline] ............ (step 2 of 4) Processing scaler, total=   0.1s
[Pipeline] ........... (step 3 of 4) Processing reducer, total=  11.8s
[Pipeline] ......... (step 4 of 4) Processing clusterer, total=   1.2s
Running pipeline for period: 90, prep type: raw
[Pipeline] ........... (step 1 of 4) Processing imputer, total=   0.1s
[Pipeline] ............ (step 2 of 4) Processing scaler, total=   0.1s
[Pipeline] ........... (step 3 of 4) Processing reducer, total=  12.1s
[Pipeline] ......... (step 4 of 4) Processing clusterer, total=   1.0s
Running pipeline for period: 90, prep type: merged
[Pipeline] ........... (step 1 of 4) Processing imputer, total=   0.2s
[Pipeline] ............ (step 2 of 4) Processing scaler, total=   0.1s
[Pipeline] ........... (step 3 of 4) Processing reducer, total=  10.7s
[Pipeline] ......... (step 4 of 4) Processing clusterer, total=   1

## Plot